# Integração entre Dados das API's do IBGE e BC

Este notebook integrará os dados recebidos pelas API's do IBGE e do BC e irá uni-los em uma só tabela

## Imports

In [2]:
import json
import pandas as pd
import requests as rqs
from pathlib import Path

## Reunir os arquivos da API do BC

Os dados da API do BC estão na pasta dados > bcb > cache em formato tipo .json

In [51]:
dados_da_api_bc = []

# Pesquisa todos os arquivos .json em dados > bcb > cache
for arquivo in Path("dados/bcb/cache").glob("*.json"):
    with open(arquivo, encoding="utf-8") as f:
        cache = json.load(f)
    dados_da_api_bc.append(pd.DataFrame(cache["dados"]["value"]))

sicor = pd.concat(dados_da_api_bc, ignore_index=True)

## Rodar o código da API_IBGE

Rodando o código com o %run e colocando o %%capture para não dar print em nada na tela, ou seja, apenas rodar e carregar todas as tabelas do API_IBGE

In [4]:
%%capture
%run API_IBGE.ipynb

## Padronizando os dados da API_BC

Padronizando os dados da API_BC para igualar os valores dos dados da API_IBGE, então, coloca-se as colunas e os valores das colunas iguais.

In [68]:
# Renomeando as colunas com o mesmo nome das colunas da API_IBGE
sicor = sicor.rename(columns = {'nomeProduto':'cultura', 'AnoEmissao':'ano', 'mesEmissao':'mes', 'Municipio':'municipio' })

# Padronizando o nome da cultura
sicor['cultura'] = sicor['cultura'].replace({
    '"SOJA"':'Soja (em grão)',
    '"CANA-DE-ALÚCAR"' :'Cana-de-açúcar',
    '"MILHO"':'Milho (em grão)'
})

# Deixando o municipio em letra minúscula
sicor['municipio'] = sicor['municipio'].str.title()

sicor

,municipio,cultura,MesEmissao,ano,cdPrograma,cdSubPrograma,cdFonteRecurso,cdTipoSeguro,cdEstado,VlCusteio,cdProduto,codCadMu,Atividade,cdModalidade,codIbge,AreaCusteio
0,Jesuítas,Soja (em grão),07,2021,0999,0,0303,9,19,120515.25,6720,40785,1,01,4112751,40.29
1,Flor Da Serra Do Sul,Soja (em grão),07,2021,0001,1,0300,3,19,884242.51,6720,49982,1,01,4107850,264.96
2,Dianópolis,Soja (em grão),07,2021,0999,0,0430,3,28,354260.00,6720,12036,1,01,1707009,100.00
3,Nova Araçá,Soja (em grão),07,2021,0050,0,0505,1,24,44311.34,6720,34832,1,01,4312807,12.00
4,Iporã,Soja (em grão),07,2021,0050,0,0505,3,19,129270.72,6720,30953,1,01,4110607,42.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831985,Caseara,Soja (em grão),01,2018,0999,0,0303,9,28,1150000.00,6720,43751,1,01,1703909,564.89
831986,Presidente Bernardes,Soja (em grão),01,2018,0001,1,0300,2,27,27267.81,6720,40163,1,01,3541208,16.94
831987,Nobres,Soja (em grão),01,2018,0999,0,0201,3,14,2606343.93,6720,12988,1,01,5105903,1278.70
831988,Águas Frias,Soja (em grão),01,2018,0001,1,0300,2,25,5892.00,6720,49494,1,01,4200556,4.00


## Padronizando os dados da API_IBGE

A padronização será reunir todos os dados da API_IBGE em uma única tabela

Padronização Municipal em tabela_ibge_municipal

In [86]:
#Unindo todas as tabelas com base nas colunas iguais
tabela_ibge_municipal = df_area_municipal.merge(df_prod_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])
tabela_ibge_municipal = tabela_ibge_municipal.merge(df_rendimento_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])

#Renomendo as colunas e retirando as colunas redundantes
tabela_ibge_municipal = tabela_ibge_municipal.drop(columns=['variavel_x','unidade_x','variavel_y','unidade_y', 'variavel','unidade'])
tabela_ibge_municipal = tabela_ibge_municipal.rename(columns={
    'valor_x':'area plantada (hectares)',
    'valor_y':'quantidade produzida (toneladas)',
    'valor':'rendimento medio (kg por hectare)',
    'codigo_cultura':'codigo cultura',
    'codigo_municipio':'codigo municipio'
})

#Padronizando o municipio para ficar idêntico ao da API_BC
tabela_ibge_municipal['municipio'] = tabela_ibge_municipal['municipio'].str.split("-").str[0].str.title()